# 03 - Monte Carlo Simulation

**Phase 3 (Weeks 3-4)** of the Portuguese wildfire catastrophe loss model.

Per the PRD's Technical Decisions ("Entry point: one main.py; notebooks for
EDA only"), the simulation engine itself lives in `monte_carlo.py`, not here -
this notebook loads Phase 2's fitted parameters, calls that module, and
displays/saves the results, matching the pattern `distribution_fitting.py` /
`02_distribution_fitting.ipynb` already set for Phase 2.

Design, verified numerically before this module was built:
`docs/phase3-frequency-severity-dependence.md`. In short:

- **Frequency: Negative Binomial**, not Poisson (Phase 2 rejected Poisson
  decisively - `models/frequency_model_choice.json`).
- **Severity is simulated in hectares**, per the PRD ("the model fits burned
  area, not euro losses, and converts to euros at the end") - the Lognormal
  body and GPD tail (`models/lognormal_severity.json`, `models/pareto_tail.json`)
  are both fitted on burned area, not euros.
- **The GPD tail is capped.** Its fitted shape (xi=0.746) exceeds 0.5, so it
  has infinite theoretical variance - left uncapped, the simulation does not
  converge and can generate non-physical single-day draws (see the linked
  doc: one prototype run reached 75.7 million ha, over 8x mainland Portugal's
  land area). Two caps apply: a practical bound (5x the historical maximum
  fire-day, `monte_carlo.historical_severity_cap`) that stabilises
  convergence, and an absolute physical backstop (6.1 million ha, mainland
  Portugal's forest+shrubland+unproductive land per ICNF's IFN6,
  `monte_carlo.PHYSICAL_CEILING_HA`).
- **A year-level frailty factor** (lognormal, mean 1, Gaussian-copula-linked
  to the annual count draw) reproduces the residual frequency-severity
  dependence Phase 1 found (annual count vs. median fire-day size,
  Spearman rho=0.57, p=0.019) that an independent model misses. A
  planning-stage prototype found sigma_z~0.25 (rho=0) reproduces the
  observed annual-loss standard deviation closely; final (sigma_z, rho)
  need a joint calibration against both the variance and correlation
  targets plus a hold-out check - **not yet done, see "Run simulation"
  below**.

Success criteria (per PRD): 100,000 scenario-years, fixed seed; standard
error on VaR(95%) < 2% (VaR(99%) reported separately, expected noisier).


## Imports

In [1]:
import sys

sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

from wildfire_model import build_fire_day_events, split_train_holdout, TRAIN_YEARS
from distribution_fitting import load_processed_data, MODELS_DIR
from monte_carlo import (
    SIMULATION_DIR, N_SCENARIOS, SEED, PHYSICAL_CEILING_HA, DEFAULT_CAP_MULTIPLIER,
    load_model_params, historical_severity_cap,
    simulate_dependent_frequency_and_frailty, simulate_severities,
    run_monte_carlo, compute_risk_metrics, convergence_check,
    plot_loss_distribution, plot_qq, plot_tail_comparison, save_simulation_results,
)


## Load fitted model parameters and training data

Same training-year fire-day events Phase 2 fitted on
(`wildfire_model.build_fire_day_events` + `split_train_holdout`), needed here
to derive `tail_probability` and the practical severity cap from the actual
data rather than hardcoding them.

In [2]:
processed_df = load_processed_data()
fire_day_events = build_fire_day_events(processed_df)
train_events, holdout_events = split_train_holdout(fire_day_events)
train_area = train_events["Burned_Area_ha"].values

nb_params = load_model_params("negative_binomial_frequency.json")
lognormal_params = load_model_params("lognormal_severity.json")
gpd_params = load_model_params("pareto_tail.json")
eur_scenarios = load_model_params("severity_euro_equivalents.json")

tail_probability = gpd_params["n_exceedances"] / len(train_area)
severity_cap_ha = historical_severity_cap(train_area)

print(f"Training fire-day events: {len(train_area)} ({TRAIN_YEARS[0]}-{TRAIN_YEARS[1]})")
print(f"Frequency: Negative Binomial r={nb_params['r']:.2f}, p={nb_params['p']:.3f}, mean={nb_params['mean']:.2f}")
print(f"Severity tail: GPD shape xi={gpd_params['shape']:.3f}, threshold={gpd_params['threshold']:,.0f} ha, "
      f"tail_probability={tail_probability:.3f}")
print(f"Severity cap: {severity_cap_ha:,.0f} ha (practical, {DEFAULT_CAP_MULTIPLIER}x historical max "
      f"{train_area.max():,.0f} ha) / {PHYSICAL_CEILING_HA:,.0f} ha (physical ceiling, ICNF IFN6)")
print(f"EUR/ha scenarios: low={eur_scenarios['low']['eur_per_ha']:.2f}, "
      f"central={eur_scenarios['central']['eur_per_ha']:.2f}, high={eur_scenarios['high']['eur_per_ha']:.2f}")

observed_annual_sd_ha = train_events.groupby(train_events["Date"].dt.year)["Burned_Area_ha"].sum().std()
print(f"\nObserved training-year annual burned-area SD (calibration target): {observed_annual_sd_ha:,.0f} ha")


Training fire-day events: 884 (2009-2020)
Frequency: Negative Binomial r=10.28, p=0.122, mean=73.67
Severity tail: GPD shape xi=0.746, threshold=2,534 ha, tail_probability=0.101
Severity cap: 982,380 ha (practical, 5.0x historical max 196,476 ha) / 6,100,000 ha (physical ceiling, ICNF IFN6)
EUR/ha scenarios: low=497.36, central=2296.10, high=3167.23

Observed training-year annual burned-area SD (calibration target): 147,411 ha


## Simulation engine, risk metrics, and diagnostic plots

Implemented in `monte_carlo.py`, not here - see its docstrings
(`simulate_dependent_frequency_and_frailty`, `simulate_severities`,
`run_monte_carlo`, `compute_risk_metrics`, `convergence_check`,
`plot_loss_distribution`, `plot_qq`, `plot_tail_comparison`,
`save_simulation_results`).

## Run simulation

**Not yet run for real.** `sigma_z` and `rho` below are the planning-stage
prototype's illustrative values (see
`docs/phase3-frequency-severity-dependence.md`), not a calibrated final
answer - a joint grid search against both the observed annual SD and the
observed count/severity correlation, checked against the 2021-2025
hold-out, still needs to be done before these numbers are reported as
Phase 3 results. This cell is left commented out until that calibration is
complete, matching how this notebook stayed commented out through Phase 2
(see git history).

In [3]:
# --- Prototype values, pending joint calibration - do not treat as final ---
# sigma_z, rho = 0.25, 0.0
# eur_per_ha = eur_scenarios["central"]["eur_per_ha"]
#
# annual_losses = run_monte_carlo(
#     nb_params, lognormal_params, gpd_params, tail_probability,
#     eur_per_ha, severity_cap_ha, sigma_z=sigma_z, rho=rho,
#     n_scenarios=N_SCENARIOS, seed=SEED,
# )
# metrics = compute_risk_metrics(annual_losses)
# convergence = convergence_check(annual_losses)
# print(metrics)
# print(convergence)  # PRD success criterion: VaR_95_se_pct < 2
#
# plot_loss_distribution(annual_losses, metrics)
# plot_qq(annual_losses)
# historical_annual_losses = train_events.groupby(train_events["Date"].dt.year)["Estimated_Loss_EUR_2025"].sum().values
# plot_tail_comparison(annual_losses, historical_annual_losses)
#
# save_simulation_results(annual_losses, {**metrics, **convergence})
